In [18]:
import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as ply

# set display options to see more data
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 115)

print("Libraries loaded successfully!")

Libraries loaded successfully!


In [19]:
# Path to the file 
file_path = '/Users/abyudhka/Desktop/Project/australian-labor-market-analysis/Data/Raw/Table 3 - Employee jobs and income, by industry and geography, 2022-23.xlsx'

# See file structure
excel_file = pd.ExcelFile(file_path)
print("Available sheets:")
for i, sheet in enumerate(excel_file.sheet_names, 1):
    print(f" {i}.{sheet}")

Available sheets:
 1.Contents
 2.Table 3.1
 3.Table 3.2
 4.Table 3.3


In [20]:
# Check structure
df_original = pd.read_excel(file_path, sheet_name='Table 3.2', skiprows=7)

print(f"Original shape: {df_original.shape}")
print(f"Original columns: {df_original.columns.tolist()[:5]}")

Original shape: (349, 42)
Original columns: ['SA3', 'SA3 NAME', "Number of jobs ('000)", "Number of jobs ('000).1", "Number of jobs ('000).2"]


In [21]:
# Check SA4 codes
df = pd.read_excel(file_path, sheet_name='Table 3.1', skiprows=7)


major_cities_sa4 = df[
     df['SA4 NAME'].str.contains('Sydney|Melbourne|Brisbane|Adelaide|Perth', case=False, na=False)
][['SA4', 'SA4 NAME']]

# check
print(major_cities_sa4)

    SA4                                SA4 NAME
16  115  Sydney - Baulkham Hills and Hawkesbury
17  116                      Sydney - Blacktown
18  117           Sydney - City and Inner South
19  118                Sydney - Eastern Suburbs
20  119               Sydney - Inner South West
21  120                     Sydney - Inner West
22  121       Sydney - North Sydney and Hornsby
23  122               Sydney - Northern Beaches
24  123               Sydney - Outer South West
25  124  Sydney - Outer West and Blue Mountains
26  125                     Sydney - Parramatta
27  126                           Sydney - Ryde
28  127                     Sydney - South West
29  128                     Sydney - Sutherland
36  206                       Melbourne - Inner
37  207                  Melbourne - Inner East
38  208                 Melbourne - Inner South
39  209                  Melbourne - North East
40  210                  Melbourne - North West
41  211                  Melbourne - Out

In [22]:
#Define metropolitan SA4s for each city
metro_sa4_codes = {
    'Sydney': list(range(115, 129)),     # Sydney SA4s: 115-128
    'Melbourne': list(range(206, 214)),  # Melbourne SA4s: 206-213
    'Brisbane': list(range(301, 306)),   # Brisbane SA4s: 301-305
    'Adelaide': list(range(401, 405)),   # Adelaide SA4s: 401-404
    'Perth': list(range(503, 508))       # Perth SA4s: 503-507
 }

# flaten to a single list
all_metro_sa4_codes = []
for city, codes in metro_sa4_codes.items():
    all_metro_sa4_codes.extend(codes)

print("\nMetropolitan SA4 Codes Defined:")
for city, codes in metro_sa4_codes.items():
    print(f"{city}:{codes}")

print(f"\nTotal metropolitan SA4s: {len(all_metro_sa4_codes)}")


Metropolitan SA4 Codes Defined:
Sydney:[115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128]
Melbourne:[206, 207, 208, 209, 210, 211, 212, 213]
Brisbane:[301, 302, 303, 304, 305]
Adelaide:[401, 402, 403, 404]
Perth:[503, 504, 505, 506, 507]

Total metropolitan SA4s: 36


In [23]:
## Create SA3 filter from SA4 codes 
# create function to filter SA3 codes
def get_sa4_from_sa3(sa3_code): # define function to accept input sa3_code
    """Extract parent SA4 code from SA3 code"""  # first 3 digits of SA3 = SA4 code
    try: # error handleing 
        sa3_str = str(int(sa3_code)) # convert to integer first and then to strings 
        if len(sa3_str) >= 3: # check string length
            return int(sa3_str[:3]) # string slicing to first 3 characters and convert to int
        return None # return nan if conditions not met
    except:
        return None # return nan if met with error in try

def is_metro_sa3(sa3_code):
    """Check if SA3 belongs to a metropolitan SA4"""
    sa4 = get_sa4_from_sa3(sa3_code) # calls function 1
    if sa4 is None:
        return False
    return sa4 in all_metro_sa4_codes # checks if sa4 values are present in all_metro_sa4_codes

def get_city_from_sa3(sa3_code):
    """Get city name from SA3 code (via parent SA4)"""
    sa4 = get_sa4_from_sa3(sa3_code)
    if sa4 is None:
        return 'Other'
      # Map SA4 to city
    if sa4 in metro_sa4_codes['Adelaide']:
        return 'Adelaide'
    elif sa4 in metro_sa4_codes['Sydney']:
        return 'Sydney'
    elif sa4 in metro_sa4_codes['Melbourne']:
        return 'Melbourne'
    elif sa4 in metro_sa4_codes['Brisbane']:
        return 'Brisbane'
    elif sa4 in metro_sa4_codes['Perth']:
        return 'Perth'
    else:
        return 'Other'

print(" SA3 Filter Functions Created")
print("\nExample usage:")
print(f"  SA3 40101 → SA4 {get_sa4_from_sa3(40101)} → City: {get_city_from_sa3(40101)}")
print(f"  SA3 40501 → SA4 {get_sa4_from_sa3(40501)} → City: {get_city_from_sa3(40501)} (should be 'Other' - regional)")
print(f"  SA3 11703 → SA4 {get_sa4_from_sa3(11703)} → City: {get_city_from_sa3(11703)}")

    

 SA3 Filter Functions Created

Example usage:
  SA3 40101 → SA4 401 → City: Adelaide
  SA3 40501 → SA4 405 → City: Other (should be 'Other' - regional)
  SA3 11703 → SA4 117 → City: Sydney


In [24]:
# Load Table 3.2
df_sa3 = pd.read_excel(file_path, sheet_name='Table 3.2', skiprows=7)
print("Loading Table 3.2...")
print(f"Original shape: {df_sa3.shape}")

# Create clean numeric sa3_code column
df_sa3['sa3_code'] = pd.to_numeric(df_sa3['SA3'], errors='coerce')

# Remove summary rows (where sa3_code is NaN)
df_sa3 = df_sa3[df_sa3['sa3_code'].notna()].copy()

# Convert to integer
df_sa3['sa3_code'] = df_sa3['sa3_code'].astype(int)
print(f"After removing summary rows: {df_sa3.shape}")

# Filter for metropolitan SA3s using the functions from Step 2
df_metro = df_sa3[df_sa3['sa3_code'].apply(is_metro_sa3)].copy()
print(f"After filtering for metro areas: {df_metro.shape}")

# add city column
df_metro['city'] = df_metro['sa3_code'].apply(get_city_from_sa3)

# droping SA3 column 
df_metro = df_metro.drop(columns=['SA3'])

# Verification
print("\nTable 3.2 Filtered for Metropolitan Areas")
print("\nCity distribution:")
print(df_metro['city'].value_counts())

print(f"\nSample data:")
print(df_metro[['sa3_code', 'SA3 NAME', 'city']].head(10))

# columns in df_metro
print(f"\n Columns list :{df_metro.columns.tolist()}")

# reorder columns
df_metro = df_metro[['sa3_code', 'SA3 NAME', 'city'] + 
                     [col for col in df_metro.columns if 'Number of jobs' in col] +
                     [col for col in df_metro.columns if 'Median employee income' in col]]

print(f" Reordered! First 5 columns: {df_metro.columns[:5].tolist()}")


Loading Table 3.2...
Original shape: (349, 42)
After removing summary rows: (336, 43)
After filtering for metro areas: (143, 43)

Table 3.2 Filtered for Metropolitan Areas

City distribution:
city
Sydney       45
Melbourne    38
Brisbane     21
Perth        20
Adelaide     19
Name: count, dtype: int64

Sample data:
    sa3_code                             SA3 NAME    city
49     11501                       Baulkham Hills  Sydney
50     11502               Dural - Wisemans Ferry  Sydney
51     11503                           Hawkesbury  Sydney
52     11504           Rouse Hill - McGraths Hill  Sydney
53     11601                            Blacktown  Sydney
54     11602                    Blacktown - North  Sydney
55     11603                         Mount Druitt  Sydney
56     11701                               Botany  Sydney
57     11702  Marrickville - Sydenham - Petersham  Sydney
58     11703                    Sydney Inner City  Sydney

 Columns list :['SA3 NAME', "Number of jobs 

In [25]:
# create industry list
industries = [
    'Agriculture, forestry and fishing',
    'Mining',
    'Manufacturing',
    'Electricity, gas, water and waste services',
    'Construction',
    'Wholesale trade',
    'Retail trade',
    'Accommodation and food services',
    'Transport, postal and warehousing',
    'Information media and telecommunications',
    'Finance and insurance services',
    'Rental, hiring and real estate services',
    'Professional, scientific and technical services',
    'Administrative and support services',
    'Public administration and safety',
    'Education and training',
    'Health care and social assistance',
    'Arts and recreation services',
    'Other services',
    'Total'
]

print(f"   Total industries: {len(industries)}")
print("\nIndustries:")
for i, industry in enumerate(industries):
    print(f"   {i}: {industry}")

# cheat new column names
new_columns = ['sa3_code', 'sa3_name', 'city']

# Add jobs columns (first 20 data columns)
for ind in industries:
    # Simplify industry name
    clean_name = ind.split(',')[0].lower().replace(' ', '_').replace('and', '').replace('__', '_').strip('_')
    new_columns.append(f'jobs_{clean_name}')

# Add income columns
for ind in industries:
    clean_name = ind.split(',')[0].lower().replace(' ', '_').replace('and', '').replace('__','_').strip('_')
    new_columns.append(f'income_{clean_name}')

print(f"new_columns count: {len(new_columns)}")  # Should be 43
print(f"df_metro columns: {len(df_metro.columns)}")  # Should be 43

# Assign to dataframe
df_metro.columns = new_columns

# Define geographic columns
geo_columns = ['sa3_code', 'sa3_name', 'city']

# Create jobs and income columns lists
jobs_columns = [col for col in df_metro.columns if col.startswith('jobs_')]
income_columns = [col for col in df_metro.columns if col.startswith('income_')]

# Separate into dataframes
df_jobs = df_metro[geo_columns + jobs_columns].copy()
df_income = df_metro[geo_columns + income_columns].copy()

print(f"\ndf_jobs shape: {df_jobs.shape}")  # Should be (143, 23) or similar
print(f"df_income shape: {df_income.shape}")  # Should be (143, 23) or similar

# Melt jobs from wide to long
df_jobs_long = df_jobs.melt(
    id_vars=geo_columns,
    var_name='industry',
    value_name='number_of_jobs'
)

# Melt income from wide to long
df_income_long = df_income.melt(
    id_vars=geo_columns,
    var_name='industry',
    value_name='median_income_dollars'
)

# Remove 'jobs_' prefix from jobs dataframe
df_jobs_long['industry'] = df_jobs_long['industry'].str.replace('jobs_', '')

# Remove 'income_' prefix from INCOME dataframe (FIX THIS!)
df_income_long['industry'] = df_income_long['industry'].str.replace('income_', '') 

print("\n Jobs dataframe after melt:")
print(df_jobs_long.head(5))

print("\n Income dataframe after melt:")
print(df_income_long.head(5))


   Total industries: 20

Industries:
   0: Agriculture, forestry and fishing
   1: Mining
   2: Manufacturing
   3: Electricity, gas, water and waste services
   4: Construction
   5: Wholesale trade
   6: Retail trade
   7: Accommodation and food services
   8: Transport, postal and warehousing
   9: Information media and telecommunications
   10: Finance and insurance services
   11: Rental, hiring and real estate services
   12: Professional, scientific and technical services
   13: Administrative and support services
   14: Public administration and safety
   15: Education and training
   16: Health care and social assistance
   17: Arts and recreation services
   18: Other services
   19: Total
new_columns count: 43
df_metro columns: 43

df_jobs shape: (143, 23)
df_income shape: (143, 23)

 Jobs dataframe after melt:
   sa3_code                    sa3_name    city     industry number_of_jobs
0     11501              Baulkham Hills  Sydney  agriculture          0.246
1     11502   

In [26]:
# merge jobs and income dataframes
df_merged = df_jobs_long.merge(
    df_income_long,
    on=['sa3_code', 'sa3_name', 'city', 'industry'],
    how='inner'
)

# convert jobs from thousnads to actual numbers 
df_merged['number_of_jobs'] = pd.to_numeric(df_merged['number_of_jobs'], errors='coerce')
df_merged['number_of_jobs'] = df_merged['number_of_jobs']*1000
df_merged['number_of_jobs'] = df_merged['number_of_jobs'].round(0).astype('Int64')

# Ensuring data type of income column
df_merged['median_income_dollars'] = pd.to_numeric(df_merged['median_income_dollars'], errors='coerce')
df_merged['median_income_dollars'] = df_merged['median_income_dollars'].round(0).astype('Int64')

# check result
print("After  merge:")
print("Shape:", df_merged.shape)       
print("\nFirst few rows:")
print(df_merged.head())
print("\nColumns:")
print(df_merged.columns.tolist())


After  merge:
Shape: (2860, 6)

First few rows:
   sa3_code                    sa3_name    city     industry  number_of_jobs  \
0     11501              Baulkham Hills  Sydney  agriculture             246   
1     11502      Dural - Wisemans Ferry  Sydney  agriculture             247   
2     11503                  Hawkesbury  Sydney  agriculture             557   
3     11504  Rouse Hill - McGraths Hill  Sydney  agriculture             191   
4     11601                   Blacktown  Sydney  agriculture             306   

   median_income_dollars  
0                  22104  
1                  27364  
2                  20466  
3                  21800  
4                  15182  

Columns:
['sa3_code', 'sa3_name', 'city', 'industry', 'number_of_jobs', 'median_income_dollars']


In [27]:
# final clean 
# rename columns
df_merged = df_merged.rename(columns={'sa3_code':'geography_code'})
df_merged = df_merged.rename(columns={'sa3_name':'geography_name'})
df_merged = df_merged.rename(columns={'industry':'industry_name'})

# reorder columns
df_merged = df_merged[['city', 'geography_code', 'geography_name', 'industry_name', 'number_of_jobs', 'median_income_dollars']]

# sort table
df_merged = df_merged.sort_values(['city', 'geography_code', 'industry_name'])

# Check table
print(" Final Table shape:", df_merged.shape)
print(f"Columns: {df_merged.columns.tolist()}")
print("Metro regions")
print("\nFirst few rows:")
print(df_merged.head())



 Final Table shape: (2860, 6)
Columns: ['city', 'geography_code', 'geography_name', 'industry_name', 'number_of_jobs', 'median_income_dollars']
Metro regions

First few rows:
          city  geography_code geography_name  \
1105  Adelaide           40101  Adelaide City   
1963  Adelaide           40101  Adelaide City   
104   Adelaide           40101  Adelaide City   
2535  Adelaide           40101  Adelaide City   
676   Adelaide           40101  Adelaide City   

                        industry_name  number_of_jobs  median_income_dollars  
1105      accommodation_food_services            5751                   6164  
1963  administrative_support_services            3107                   7190  
104                       agriculture             281                   6129  
2535         arts_recreation_services             971                   8510  
676                      construction             679                  31025  


In [28]:
## Quality checks
df_metro_regions_final = df_merged.copy()

# Check for missing values (NaN)
print("\nMissing values check:")
print(df_metro_regions_final.isnull().sum())

# Verify all metro regions are in final data
print("\nMetro regions check:",df_metro_regions_final['geography_name'].nunique())
print(df_metro_regions_final['geography_name'].unique())


Missing values check:
city                      0
geography_code            0
geography_name            0
industry_name             0
number_of_jobs           19
median_income_dollars    19
dtype: int64

Metro regions check: 143
['Adelaide City' 'Adelaide Hills' 'Burnside' 'Campbelltown (SA)'
 'Norwood - Payneham - St Peters' 'Prospect - Walkerville' 'Unley'
 'Gawler - Two Wells' 'Playford' 'Port Adelaide - East' 'Salisbury'
 'Tea Tree Gully' 'Holdfast Bay' 'Marion' 'Mitcham' 'Onkaparinga'
 'Charles Sturt' 'Port Adelaide - West' 'West Torrens' 'Capalaba'
 'Cleveland - Stradbroke' 'Wynnum - Manly' 'Bald Hills - Everton Park'
 'Chermside' 'Nundah' 'Sandgate' 'Carindale' 'Holland Park - Yeronga'
 'Mt Gravatt' 'Nathan' 'Rocklea - Acacia Ridge' 'Sunnybank' 'Centenary'
 'Kenmore - Brookfield - Moggill' 'Sherwood - Indooroopilly'
 'The Gap - Enoggera' 'Brisbane Inner' 'Brisbane Inner - East'
 'Brisbane Inner - North' 'Brisbane Inner - West' 'Brunswick - Coburg'
 'Darebin - South' 'Essendon' 

In [31]:
## DEBUGGING: Check for data issues

print("="*80)
print("DATA QUALITY INVESTIGATION")
print("="*80)

# 1. Check for missing values
print("\n1. MISSING VALUES:")
missing_jobs = df_merged['number_of_jobs'].isna().sum()
missing_income = df_merged['median_income_dollars'].isna().sum()
print(f"   Missing jobs: {missing_jobs}")
print(f"   Missing income: {missing_income}")

if missing_jobs > 0 or missing_income > 0:
    print("\n   Rows with missing data:")
    missing_rows = df_merged[df_merged['number_of_jobs'].isna() | df_merged['median_income_dollars'].isna()]
    print(missing_rows[['geography_name', 'industry_name', 'number_of_jobs', 'median_income_dollars']])

# 2. Check income value ranges
print("\n2. INCOME VALUE RANGES:")
print(f"   Min income: ${df_merged['median_income_dollars'].min():,.0f}")
print(f"   Max income: ${df_merged['median_income_dollars'].max():,.0f}")
print(f"   Median income: ${df_merged['median_income_dollars'].median():,.0f}")

# 3. Sample of income values
print("\n3. SAMPLE INCOME VALUES (Adelaide City):")
adelaide_sample = df_merged[df_merged['geography_name'] == 'Adelaide City'].head(10)
print(adelaide_sample[['industry_name', 'number_of_jobs', 'median_income_dollars']])

# 4. Industries with suspiciously low income (<$20,000)
print("\n4. INDUSTRIES WITH INCOME < $20,000:")
low_income = df_merged[df_merged['median_income_dollars'] < 20000]
print(f"   Count: {len(low_income)} rows")
if len(low_income) > 0:
    print(low_income[['geography_name', 'industry_name', 'median_income_dollars']].head(20))

DATA QUALITY INVESTIGATION

1. MISSING VALUES:
   Missing jobs: 19
   Missing income: 19

   Rows with missing data:
              geography_name                         industry_name  \
1031  Blue Mountains - South           accommodation_food_services   
1889  Blue Mountains - South       administrative_support_services   
30    Blue Mountains - South                           agriculture   
2461  Blue Mountains - South              arts_recreation_services   
602   Blue Mountains - South                          construction   
2175  Blue Mountains - South                    education_training   
459   Blue Mountains - South                           electricity   
1460  Blue Mountains - South            finance_insurance_services   
2318  Blue Mountains - South         health_care_social_assistance   
1317  Blue Mountains - South  information_media_telecommunications   
316   Blue Mountains - South                         manufacturing   
173   Blue Mountains - South               

In [38]:
# Fix the missing values
df_metro_regions_final['number_of_jobs'] = df_metro_regions_final['number_of_jobs'].fillna(0)
df_metro_regions_final['median_income_dollars'] = df_metro_regions_final['median_income_dollars'].fillna(0)

# Verify
print("Missing values after fix:")
print(df_metro_regions_final.isnull().sum())
print("Sample dataset")
print(df_metro_regions_final.groupby('city').head(3))

Missing values after fix:
city                     0
geography_code           0
geography_name           0
industry_name            0
number_of_jobs           0
median_income_dollars    0
dtype: int64
Sample dataset
           city  geography_code         geography_name  \
1105   Adelaide           40101          Adelaide City   
1963   Adelaide           40101          Adelaide City   
104    Adelaide           40101          Adelaide City   
1084   Brisbane           30101               Capalaba   
1942   Brisbane           30101               Capalaba   
83     Brisbane           30101               Capalaba   
1046  Melbourne           20601     Brunswick - Coburg   
1904  Melbourne           20601     Brunswick - Coburg   
45    Melbourne           20601     Brunswick - Coburg   
1124      Perth           50301  Cottesloe - Claremont   
1982      Perth           50301  Cottesloe - Claremont   
123       Perth           50301  Cottesloe - Claremont   
1001     Sydney           1150

In [39]:
## Final clean and export
# Sort 
df_metro_regions_final = df_metro_regions_final.sort_values(
    ['city', 'geography_code', 'industry_name']
).reset_index(drop=True)

print("Final sorted data:")
print(df_metro_regions_final.head(10))

# Save to processed folder
output_path = '/Users/abyudhka/Desktop/Project/australian-labor-market-analysis/Data/processed/jobs_&_income_by_industry_geography_2023.csv'

df_metro_regions_final.to_csv(output_path, index=False)
print(f" Saved to: {output_path}")
print(f"   Rows: {len(df_metro_regions_final)}")
print(f"   Columns: {len(df_metro_regions_final.columns)}")

# Verification: Read back
df_test = pd.read_csv(output_path)
print(f"\n Verification: Successfully read back {len(df_test)} rows")
print("\nFile preview:")
print(df_test.head())

Final sorted data:
       city  geography_code geography_name  \
0  Adelaide           40101  Adelaide City   
1  Adelaide           40101  Adelaide City   
2  Adelaide           40101  Adelaide City   
3  Adelaide           40101  Adelaide City   
4  Adelaide           40101  Adelaide City   
5  Adelaide           40101  Adelaide City   
6  Adelaide           40101  Adelaide City   
7  Adelaide           40101  Adelaide City   
8  Adelaide           40101  Adelaide City   
9  Adelaide           40101  Adelaide City   

                          industry_name  number_of_jobs  median_income_dollars  
0           accommodation_food_services            5751                   6164  
1       administrative_support_services            3107                   7190  
2                           agriculture             281                   6129  
3              arts_recreation_services             971                   8510  
4                          construction             679              